# AI应用基础 · 工单M2.1《工业焊接缺陷检测系统》
**第6周 | 2课时(80分钟) | 理实一体**  | 学生实训版

| 层 | 任务定位 | 达标要求 |
|----|---------|---------|
| **A 师傅** | 进阶拓展 | 加数据增强（随机翻转/平移）对比训练曲线，解释增强为何有效；示范讲解 |
| **B 组长** | 标准达成 | ≥0.88；数据增强对比+loss曲线 |
| **C 操作员** | 基础完成 | 准确率≥0.80；形状追踪表基本正确 |
| **D 学徒** | 跟做保底 | 跟做示例跑通TODO-1, 程序出结果 |
> 每4周按工单成绩动态升降层（连续2次≥B升一层）


## 一、课前资讯（Predict·预测）
> **预习快问**：卷积核3×3、输入64×64、不加padding，输出尺寸是多少？

> **课前检查**：导学单+预习自测已完成；data/数据已就位、starter可运行（--check环境OK）

## 二、PRIMM五步法（先读后写）
本工单按国际编程教学法 **PRIMM** 组织：
1. **Predict 预测**——先运行下方示例，猜输出再验证；
2. **Run 运行**——跑通环境与数据；
3. **Investigate 研究**——读懂starter每段在做什么；
4. **Modify 修改**——完成TODO改造；
5. **Make 创造**——独立完成工单任务。


In [ ]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())

subprocess.run([sys.executable, "数据与工具/generate_data.py", "m2img"], check=True)
print("数据已生成: data/")

## 三、任务要求（Make）
**实施步骤**
1. 加载npy数据并归一化(/255)+划分
2. 补全forward与训练循环(3 epochs)
3. 记录每轮loss与测试准确率
4. 输出2张测试图预测+置信度

**验收标准**：及格(D)=`准确率≥0.80；形状追踪表基本正确` ；优秀(B+)=`≥0.88；数据增强对比+loss曲线`


### Starter 1：`# -*- coding: utf-8 -*-`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
# -*- coding: utf-8 -*-
"""M2.1 工业焊接缺陷检测(tiny-CNN, CPU约2分钟) —— 学生代码框架"""
import json
import os

import numpy as np
import torch
import torch.nn as nn

ROOT = "data/weld_images"
CLASSES = ["normal", "porosity", "slag", "crack"]

### Starter 2：`class TinyCNN(nn.Module):`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
class TinyCNN(nn.Module):
    """2卷积+2全连接; 完成TODO-1的forward"""

    def __init__(self, num_classes=4):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, 3)    # 64→62
        self.pool = nn.MaxPool2d(2, 2)     # 62→31
        self.conv2 = nn.Conv2d(8, 16, 3)   # 31→29 →pool→14
        self.fc1 = nn.Linear(16 * 14 * 14, 64)
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, x):
        """TODO-1: conv1→relu→pool→conv2→relu→pool→flatten→fc1→relu→fc2"""
        # === 你的代码 ===
        raise NotImplementedError

### Starter 3：`def load_dataset():`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
def load_dataset():
    with open(os.path.join(ROOT, "index.json"), encoding="utf-8") as f:
        meta = json.load(f)
    X = np.stack([np.load(os.path.join(ROOT, m["file"])) for m in meta])
    y = np.array([CLASSES.index(m["label"]) for m in meta])
    X = X.astype(np.float32) / 255.0          # 归一化(勿忘!)
    X = X[:, None, :, :]                      # (N,1,64,64) 通道维
    idx = np.random.RandomState(42).permutation(len(X))
    X, y = X[idx], y[idx]
    n_train = int(len(X) * 0.8)
    return (X[:n_train], y[:n_train]), (X[n_train:], y[n_train:])

### Starter 4：`def main():`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
def main():
    (Xtr, ytr), (Xte, yte) = load_dataset()
    print("形状追踪: 输入", Xtr.shape, "(N,C,H,W)")
    device = "cpu"
    model = TinyCNN().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1.5e-3)
    lossf = nn.CrossEntropyLoss()

    xt = torch.tensor(Xte)
    yt = torch.tensor(yte)
    for epoch in range(6):
        model.train()
        # TODO-2: 小批量训练(batch=32): 前向→loss→反向→step, 累计打印epoch loss
        # === 你的代码 ===
        model.eval()
        with torch.no_grad():
            pred = model(xt).argmax(1)
            acc = (pred == yt).float().mean().item()
        print(f"epoch {epoch + 1}  测试准确率={acc:.3f}")

    # TODO-3: 任取2张测试图, 输出预测类别与softmax置信度
    # TODO-4: (优秀)数据增强: 随机水平翻转后再训练, 对比准确率
    # TODO-5: (优秀)绘制loss曲线与混淆矩阵(matplotlib)

### Starter 5：`if __name__ == "__main__":`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
if __name__ == "__main__":
    main()

## 四、提交要求与避坑
- 提交: `学号_姓名_M2.1.py` 或本notebook(含输出)
- ⚠ 输入必须/255归一化，否则loss居高不下
- ⚠ PyTorch通道序是N,C,H,W——numpy转tensor后要补通道维
- ⚠ 测试集指标为准，禁止在测试集上挑图刷准确率